# From Agent Loop to Agent Harness

**What you will build:** a small, testable agent harness in pure Python. It has a provider boundary, canonical turns, a tool registry, declarative risk, a separate permission policy, domain events, a rich transcript, and a model-facing feed. Read-only tools run concurrently; effects stay ordered; every tool call receives a result.

This is the capstone for Block 0. The loop from 0.3 remains the center. The new code is the engineering around it that lets one loop serve a notebook, CLI, API, or desktop app.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/09_from_agent_loop_to_harness.ipynb), or run locally in Jupyter. The main path is deterministic and offline; the optional live section uses OpenRouter.

In [ ]:
# Setup — the harness itself uses only the standard library.
# openai is installed for the optional live-provider adapter near the end.
!pip install -q openai

import asyncio
import getpass
import json
import os
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable, Protocol

print("Ready — the main demo makes no network calls.")

## The loop is small; the harness owns the boundaries

The loop you wrote in 0.3 answers one question: **what happens after the model requests a tool?** A harness must answer several questions without mixing them together:

| Responsibility | Question |
|---|---|
| Provider adapter | How does this model represent text and tool calls? |
| Tool registry | Which tools exist, what are their schemas, and how do they run? |
| Permission policy | Should this particular request run? |
| Executor / isolation | What can the process actually touch if it runs? |
| Transcript | What happened, including UI and audit metadata? |
| Provider feed | What subset of that history should the model see? |
| Events | How do a CLI, notebook, or UI observe the run? |
| Invariants | What must remain true after denial, error, interruption, or a limit? |

```text
surface ──▶ TurnEngine ──▶ Provider
              │   ▲
              │   └──── canonical AssistantTurn
              ├──▶ PermissionPolicy ──▶ ToolRegistry
              ├──▶ Transcript ──▶ provider_messages()
              └──▶ Event stream ──▶ surface
```

This is not a framework yet. It is a set of small ownership boundaries you can inspect.

## 1. Canonical types: the center does not speak vendor dialects

OpenAI, Anthropic and Gemini serialize tool calls differently. The engine should not know. Each provider adapter converts its wire format into the same `AssistantTurn`; the engine works only with these types:

In [ ]:
class Risk(str, Enum):
    READ = "read"
    WRITE = "write"
    EXEC = "exec"
    EXTERNAL = "external"


@dataclass(frozen=True)
class ToolCall:
    id: str
    name: str
    arguments: dict[str, Any]


@dataclass
class AssistantTurn:
    content: str = ""
    tool_calls: list[ToolCall] = field(default_factory=list)


@dataclass(frozen=True)
class Event:
    type: str
    data: dict[str, Any]


class Provider(Protocol):
    async def complete(
        self, messages: list[dict], tools: list[dict]
    ) -> AssistantTurn:
        ...

A deliberately small protocol is a useful constraint. **Providers do not own the loop.** They make one model call and return one canonical turn. Retries for HTTP compatibility may live in an adapter; model→tool→model iteration belongs to the runtime.

In [ ]:
class ScriptedProvider:
    """Deterministic provider for demos and trajectory tests."""

    def __init__(self, turns: list[AssistantTurn]):
        self.turns = list(turns)
        self.seen_messages: list[list[dict]] = []

    async def complete(self, messages, tools) -> AssistantTurn:
        self.seen_messages.append(messages)
        if not self.turns:
            raise RuntimeError("Script exhausted before the harness stopped")
        return self.turns.pop(0)


class OpenRouterProvider:
    """Optional live adapter: OpenAI wire format in, canonical turn out."""

    def __init__(self, client, model: str):
        self.client = client
        self.model = model

    async def complete(self, messages, tools) -> AssistantTurn:
        kwargs = {
            "model": self.model,
            "messages": messages,
            "temperature": 0,
        }
        if tools:
            kwargs["tools"] = tools

        response = await asyncio.to_thread(
            self.client.chat.completions.create, **kwargs
        )
        message = response.choices[0].message
        calls = []
        for call in message.tool_calls or []:
            try:
                arguments = json.loads(call.function.arguments)
            except json.JSONDecodeError as exc:
                arguments = {"_invalid_json": call.function.arguments,
                             "_parse_error": str(exc)}
            calls.append(ToolCall(call.id, call.function.name, arguments))
        return AssistantTurn(message.content or "", calls)

The adapter is the right place for provider-specific compatibility: signed thinking blocks, thought signatures, schema restrictions, reasoning parameters, malformed tool arguments, or a different streaming protocol. Keeping that work at the edge lets the rest of the harness stay stable.

## 2. Tool registry: capability plus metadata

A tool has two audiences. The **model** needs a name, description and JSON Schema. The **runtime** needs a callable and intrinsic metadata such as risk. Registration joins those views; policy remains separate:

In [ ]:
@dataclass
class ToolSpec:
    name: str
    description: str
    parameters: dict[str, Any]
    handler: Callable[..., Any]
    risk: Risk = Risk.READ


class ToolRegistry:
    def __init__(self):
        self._tools: dict[str, ToolSpec] = {}

    def register(self, spec: ToolSpec) -> None:
        if spec.name in self._tools:
            raise ValueError(f"Duplicate tool: {spec.name}")
        self._tools[spec.name] = spec

    def get(self, name: str) -> ToolSpec:
        if name not in self._tools:
            raise KeyError(f"Unknown tool: {name}")
        return self._tools[name]

    def find(self, name: str) -> ToolSpec | None:
        """Look up metadata without turning an unknown tool into an exception."""
        return self._tools.get(name)

    def schemas(self) -> list[dict]:
        return [
            {"type": "function", "function": {
                "name": spec.name,
                "description": spec.description,
                "parameters": spec.parameters,
            }}
            for spec in self._tools.values()
        ]

    async def execute(self, call: ToolCall) -> str:
        try:
            spec = self.get(call.name)
            result = await asyncio.to_thread(spec.handler, **call.arguments)
            return str(result)
        except Exception as exc:
            # Tool failure is data for the next model turn, not a broken loop.
            return f"Error: {type(exc).__name__}: {exc}"

## 3. Permission policy: decision is not execution

The policy classifies a proposed call. It does not prompt the user and does not run the tool. That makes the decision testable without a UI.

Remember 0.7's security boundary: approval answers **whether** to run; isolation limits **what the process can do**. A `Risk.EXEC` tool still needs a container, VM, or OS sandbox after the user approves it.

In [ ]:
@dataclass(frozen=True)
class Decision:
    allowed: bool
    needs_user: bool = False
    reason: str = ""


class PermissionPolicy:
    def __init__(self, preapproved: set[str] | None = None):
        self.preapproved = set(preapproved or ())

    def evaluate(self, spec: ToolSpec, arguments: dict) -> Decision:
        if spec.risk is Risk.READ:
            return Decision(True, reason="read-only")
        if spec.name in self.preapproved:
            return Decision(True, reason="explicit preapproval")
        return Decision(False, needs_user=True,
                        reason=f"{spec.risk.value} action requires approval")

## 4. Transcript: persist richly, send selectively

The transcript belongs to the application. It can retain timestamps and display metadata for a UI or audit trail. `provider_messages()` is the single exit to a model, so display-only sidecars cannot leak accidentally:

In [ ]:
DISPLAY_ONLY = {"ts", "source", "_display", "reasoning", "audit_id"}


class Transcript:
    def __init__(self):
        self.messages: list[dict] = []

    def append(self, message: dict) -> None:
        self.messages.append({
            **message,
            "ts": datetime.now(timezone.utc).isoformat(),
        })

    def add_tool_result(
        self, call: ToolCall, result: str, display: dict | None = None
    ) -> None:
        message = {
            "role": "tool",
            "tool_call_id": call.id,
            "content": result,
        }
        if display:
            message["_display"] = display
        self.append(message)

    def provider_messages(self) -> list[dict]:
        return [
            {key: value for key, value in message.items()
             if key not in DISPLAY_ONLY}
            for message in self.messages
        ]

## 5. TurnEngine: events around the same old loop

The engine below adds four production-shaped rules:

1. Authorization happens sequentially before execution.
2. Approved reads can run concurrently; writes and external effects stay ordered.
3. Denials, unknown tools and exceptions still become tool results.
4. Surfaces observe domain events instead of being hard-coded into the engine.

The approver is injected just like the provider. In a notebook it can be a function; in a web app it might suspend the run and wait for a WebSocket response.

In [ ]:
def assistant_message(turn: AssistantTurn) -> dict:
    message: dict[str, Any] = {
        "role": "assistant",
        "content": turn.content or None,
    }
    if turn.tool_calls:
        message["tool_calls"] = [
            {
                "id": call.id,
                "type": "function",
                "function": {
                    "name": call.name,
                    "arguments": json.dumps(call.arguments),
                },
            }
            for call in turn.tool_calls
        ]
    return message


class TurnEngine:
    def __init__(
        self,
        provider: Provider,
        registry: ToolRegistry,
        policy: PermissionPolicy,
        approver: Callable[[ToolCall, ToolSpec], bool] | None = None,
        max_iterations: int = 8,
    ):
        self.provider = provider
        self.registry = registry
        self.policy = policy
        self.approver = approver or (lambda call, spec: False)
        self.max_iterations = max_iterations
        self.transcript = Transcript()
        self.interrupted = False

    def request_interrupt(self) -> None:
        self.interrupted = True

    async def run(self, task: str):
        self.transcript.append({"role": "user", "content": task})
        yield Event("turn_start", {"task": task})

        for iteration in range(1, self.max_iterations + 1):
            if self.interrupted:
                yield Event("interrupted", {"iteration": iteration})
                return

            turn = await self.provider.complete(
                self.transcript.provider_messages(),
                self.registry.schemas(),
            )
            self.transcript.append(assistant_message(turn))
            if turn.content:
                yield Event("assistant_message", {"content": turn.content})

            if not turn.tool_calls:
                yield Event("turn_end", {
                    "status": "complete", "iterations": iteration
                })
                return

            results: dict[str, str] = {}
            approved: list[tuple[ToolCall, ToolSpec]] = []

            # Ask in model order. Several simultaneous prompts make poor UX.
            for call in turn.tool_calls:
                yield Event("tool_proposed", {
                    "id": call.id, "name": call.name,
                    "arguments": call.arguments,
                })
                try:
                    spec = self.registry.get(call.name)
                except KeyError as exc:
                    results[call.id] = f"Error: {exc}"
                    continue

                decision = self.policy.evaluate(spec, call.arguments)
                allowed = decision.allowed
                if decision.needs_user:
                    yield Event("permission_required", {
                        "id": call.id,
                        "name": call.name,
                        "reason": decision.reason,
                    })
                    allowed = self.approver(call, spec)

                if not allowed:
                    results[call.id] = f"Denied: {decision.reason}"
                elif self.interrupted:
                    results[call.id] = "Error: interrupted by user"
                else:
                    approved.append((call, spec))

            reads = [(call, spec) for call, spec in approved
                     if spec.risk is Risk.READ]
            effects = [(call, spec) for call, spec in approved
                       if spec.risk is not Risk.READ]

            for call, _ in approved:
                yield Event("tool_started", {"id": call.id, "name": call.name})

            # Independent reads may overlap. Gather preserves input order.
            read_values = await asyncio.gather(
                *(self.registry.execute(call) for call, _ in reads)
            )
            results.update({
                call.id: value
                for (call, _), value in zip(reads, read_values)
            })

            # Side effects remain sequential and deterministic.
            for call, _ in effects:
                results[call.id] = await self.registry.execute(call)

            # Append in the original assistant-call order, regardless of scheduling.
            for call in turn.tool_calls:
                result = results[call.id]
                spec = self.registry.find(call.name)
                display = {"risk": spec.risk.value} if spec else {"risk": "unknown"}
                self.transcript.add_tool_result(call, result, display)
                yield Event("tool_finished", {
                    "id": call.id,
                    "name": call.name,
                    "result": result,
                })

            yield Event("iteration_end", {"iteration": iteration})

        yield Event("turn_end", {
            "status": "max_iterations_exceeded",
            "iterations": self.max_iterations,
        })

Notice that the engine asks the registry for tool metadata but never reaches into its storage or invokes a handler directly. Harness design is largely deciding which component owns each fact — then making that boundary boring to use.

## Run a complete trajectory — offline

Our fake workspace is a dictionary. The scripted provider first requests two independent reads, then one write, then stops. The harness does not special-case the script; a live provider produces the same `AssistantTurn` objects.

In [ ]:
NOTES = {
    "project": "Atlas ships Friday.",
    "decision": "Use Postgres for the first release.",
}

def read_note(name: str) -> str:
    """Read one note."""
    return NOTES[name]

def write_note(name: str, content: str) -> str:
    """Create or replace one note."""
    NOTES[name] = content
    return f"Wrote {name}"


registry = ToolRegistry()
registry.register(ToolSpec(
    name="read_note",
    description="Read one project note by name.",
    parameters={"type": "object", "properties": {
        "name": {"type": "string"}
    }, "required": ["name"]},
    handler=read_note,
    risk=Risk.READ,
))
registry.register(ToolSpec(
    name="write_note",
    description="Create or replace one project note.",
    parameters={"type": "object", "properties": {
        "name": {"type": "string"},
        "content": {"type": "string"},
    }, "required": ["name", "content"]},
    handler=write_note,
    risk=Risk.WRITE,
))

provider = ScriptedProvider([
    AssistantTurn(tool_calls=[
        ToolCall("read_1", "read_note", {"name": "project"}),
        ToolCall("read_2", "read_note", {"name": "decision"}),
    ]),
    AssistantTurn(tool_calls=[
        ToolCall("write_1", "write_note", {
            "name": "brief",
            "content": "Atlas ships Friday using Postgres.",
        }),
    ]),
    AssistantTurn("Done — I read both source notes and wrote the brief."),
])

def approve_demo(call: ToolCall, spec: ToolSpec) -> bool:
    print(f"  [human approves] {spec.risk.value}: {call.name}{call.arguments}")
    return True

engine = TurnEngine(
    provider=provider,
    registry=registry,
    policy=PermissionPolicy(),
    approver=approve_demo,
)

In [ ]:
events = []
async for event in engine.run("Build a short project brief from the notes."):
    events.append(event)
    if event.type in {
        "assistant_message", "permission_required",
        "tool_finished", "turn_end",
    }:
        print(f"{event.type:20} {event.data}")

print("\nbrief:", NOTES["brief"])

The two reads were authorized first and then scheduled together. The write waited for both and required approval. The event consumer could just as easily update a terminal, send WebSocket frames, record metrics, or render tool cards.

## Inspect the two histories

The persisted transcript contains timestamps and risk labels for the surface. The provider saw neither. Tool-call/result ordering remains valid in both views:

In [ ]:
print("TRANSCRIPT TOOL RESULT:")
print(json.dumps(engine.transcript.messages[2], indent=2))

print("\nSAME RESULT IN NEXT PROVIDER REQUEST:")
print(json.dumps(provider.seen_messages[1][2], indent=2))

assert "_display" in engine.transcript.messages[2]
assert "_display" not in provider.seen_messages[1][2]
assert "ts" not in provider.seen_messages[1][2]

## Test the invariant that keeps runs resumable

A provider conversation is malformed if an assistant tool call never receives a matching tool result. Denial and failure are not reasons to omit it — they are results the model must observe.

The helper below finds orphaned calls. Then a second run denies a write and proves that the transcript is still structurally complete:

In [ ]:
def orphaned_tool_calls(messages: list[dict]) -> set[str]:
    requested = {
        call["id"]
        for message in messages
        for call in message.get("tool_calls", [])
    }
    answered = {
        message["tool_call_id"]
        for message in messages
        if message.get("role") == "tool"
    }
    return requested - answered


before = dict(NOTES)
denial_provider = ScriptedProvider([
    AssistantTurn(tool_calls=[
        ToolCall("write_denied", "write_note", {
            "name": "brief", "content": "Unapproved replacement"
        }),
    ]),
    AssistantTurn("The write was denied, so I left the brief unchanged."),
])
denial_engine = TurnEngine(
    denial_provider,
    registry,
    PermissionPolicy(),
    approver=lambda call, spec: False,
)

async for _ in denial_engine.run("Replace the brief."):
    pass

assert NOTES == before
assert orphaned_tool_calls(denial_engine.transcript.messages) == set()
denied_result = next(
    m for m in denial_engine.transcript.messages
    if m.get("tool_call_id") == "write_denied"
)
print(denied_result["content"])
print("orphaned calls:", orphaned_tool_calls(denial_engine.transcript.messages))

This small assertion is a trajectory test: it checks the shape of the whole run, not just the final sentence. Add similar tests for unknown tools, malformed arguments, max iterations, interruption, permission decisions, tool order and provider-feed filtering.

Later, PydanticAI gives you typed testing utilities and LangGraph gives you first-class checkpoints. The invariant does not change.

::::{dropdown} 🔍 Optional: use a live OpenRouter model
:color: info

The deterministic provider is the right default for learning and tests. To let a model choose the same tools, set `RUN_LIVE = True`. This cell asks for a key only then.

The approval callback remains deliberately visible. The live model may choose a different path, but it cannot bypass the registry or policy.

In [ ]:
RUN_LIVE = False

if RUN_LIVE:
    from openai import OpenAI

    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = getpass.getpass(
            "Paste your OpenRouter API key: "
        )

    MODEL = "meta-llama/llama-3.3-70b-instruct"
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )

    def approve_live(call: ToolCall, spec: ToolSpec) -> bool:
        answer = input(
            f"Approve {spec.risk.value} {call.name}{call.arguments}? [y/N] "
        )
        return answer.strip().lower() == "y"

    live_engine = TurnEngine(
        OpenRouterProvider(client, MODEL),
        registry,
        PermissionPolicy(),
        approver=approve_live,
    )
    async for event in live_engine.run(
        "Read project and decision, then write a brief note combining them."
    ):
        if event.type in {"assistant_message", "tool_finished", "turn_end"}:
            print(event.type, event.data)

## What this notebook still leaves out

A good boundary makes future work possible; it does not pretend that work disappeared.

| Production concern | Notebook version | Typical next step |
|---|---|---|
| Streaming | one complete turn | provider chunks bridged into assistant-delta events |
| Durable resume | in-memory transcript | append-only store + pending approval keyed by tool-call ID |
| Cancellation | cooperative flag | cancel provider stream and interrupt the active executor |
| Isolation | none | container, VM, or OS sandbox; network policy |
| Provider quirks | one simple adapter | adapter-private replay metadata and capability checks |
| Observability | printed events | latency, tokens, cost, errors and trace storage |
| Quality | trajectory assertions | datasets, graders and regression thresholds |

For a readable product-scale example, study [andrewyng/openworker](https://github.com/andrewyng/openworker). Its loop is still recognizable; most of its code handles these boundaries. Treat it as a case study, not a dependency or a security blueprint.

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Unknown tool.** Script a call to `delete_everything`. **Done when:** the engine appends an error result, emits `tool_finished`, and `orphaned_tool_calls(...)` stays empty.
2. **Event metrics.** Write a consumer that counts model iterations, proposed tools, approvals and errors without changing `TurnEngine`. **Done when:** it reports the denial trajectory above correctly.
3. **Durable checkpoint.** Save `transcript.messages` as JSONL after every `iteration_end`, load it into a fresh `Transcript`, and identify unanswered tool calls with the helper. **Done when:** you can explain what extra idempotency key an effectful tool needs before automatic resume.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Unknown tool:**

```python
unknown_provider = ScriptedProvider([
    AssistantTurn(tool_calls=[
        ToolCall("unknown_1", "delete_everything", {})
    ]),
    AssistantTurn("That tool is unavailable."),
])
unknown_engine = TurnEngine(
    unknown_provider, registry, PermissionPolicy()
)
async for _ in unknown_engine.run("Delete everything."):
    pass

assert orphaned_tool_calls(unknown_engine.transcript.messages) == set()
assert any(
    "Unknown tool" in message.get("content", "")
    for message in unknown_engine.transcript.messages
)
```

**2 — Event metrics:** keep the consumer outside the engine:

```python
async def event_counts(engine, task):
    counts = {}
    async for event in engine.run(task):
        counts[event.type] = counts.get(event.type, 0) + 1
    return counts
```

Use a fresh engine/provider for each run; a scripted provider consumes its turns.

**3 — Durable checkpoint:** JSONL gives you a transcript, not exactly-once execution. Before replaying an effect, persist a record keyed by `tool_call_id` and check whether that effect already completed. A crash can happen after the external side effect but before the tool result reaches the transcript — the awkward gap every durable harness must design for.
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| `Script exhausted` | The last scripted turn still requested tools | End the script with an `AssistantTurn` that has no tool calls |
| Re-running the demo fails or changes state | Providers consume turns and `NOTES` is mutable | Re-run the setup cells that create them |
| Live model calls a tool with invalid JSON | Provider returned malformed arguments | The adapter preserves the parse error as data; let the registry return an error observation |
| An approval prompt blocks in a frontend | `input()` is not a durable UI mechanism | Keep policy pure; replace the injected approver with the surface's async approval flow |
| Parallel calls scramble the transcript | Results were appended as tasks finished | Gather concurrently, then append in the assistant's original tool-call order |
| `KeyError` on `results[call.id]` | A new denial/error path forgot to create a result | Preserve the invariant: every proposed tool call gets exactly one result |
::::

**Block 0 complete.** You built the wire format, structured output, tools, workflows, reflection, human gates, memory, a coding agent, context engineering, and now the harness that composes those mechanisms. The loop never became mysterious; the ownership boundaries became explicit.

**What's next — Block 1:** PydanticAI replaces some of this plumbing with typed, maintained abstractions. In **1.0**, you will rebuild the raw agent side by side and inspect exactly what the framework owns — and what your application still must own.